# Exploratory Data Analysis (EDA) on Superstore Retail Sales
**Goal:** data wrangling, missing-value profiling, and commercial insights on purchase volume, monthly seasonality, category-wise revenue and discount sensitivity.

**Run it in Google Colab (no install needed):** upload this file at colab.research.google.com → *Runtime → Run all*.

**Workflow:** 1) Load → 2) Profile → 3) Clean → 4) Purchase volume → 5) Seasonality → 6) Category / Region / Segment → 7) Discount sensitivity → 8) Executive takeaways

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 5)
pd.options.display.float_format = "{:,.2f}".format

## 1. Load the data

In [ ]:
URL = "https://www.datheon.in/datasets/superstore_sales.csv"
LOCAL = "superstore_sales.csv"

def load():
    try:
        return pd.read_csv(URL, encoding="utf-8")
    except Exception as e:
        print("Could not download from URL:", type(e).__name__)
    if not os.path.exists(LOCAL):
        try:  # Google Colab fallback: manual upload
            from google.colab import files
            print("Please upload superstore_sales.csv")
            files.upload()
        except ImportError:
            raise FileNotFoundError(f"Download the CSV and place it next to this notebook as '{LOCAL}'")
    try:
        return pd.read_csv(LOCAL, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(LOCAL, encoding="latin-1")

raw = load()
print("Shape:", raw.shape)
raw.head()

## 2. Profile the raw data (types, missing values, duplicates)

In [ ]:
raw.info()
print("\nDuplicate rows:", raw.duplicated().sum())

missing = (raw.isna().sum().to_frame("missing_count")
           .assign(missing_pct=lambda d: d.missing_count / len(raw) * 100)
           .query("missing_count > 0"))
print("\nMissing-value profile:")
display(missing if len(missing) else "No missing values found")
raw.describe().T

## 3. Data cleaning
Decisions (each one is deliberate):
| Issue | Action | Why |
|---|---|---|
| Column-name whitespace | strip | safe referencing |
| Duplicate rows | drop | avoid double counting revenue |
| `Order Date` / `Ship Date` stored as text | parse to datetime (`errors="coerce"`) | enables time analysis |
| Missing `Order Date`, `Sales`, `Profit` | drop row | core metrics; imputing would distort revenue |
| Missing `Discount` | fill with 0 | no recorded discount = full price |
| Missing `Quantity` | fill with median | minor attribute |
| Missing categorical fields | fill with `"Unknown"` | keeps the row for other analyses |
| `Postal Code` loses leading zeros (e.g. 6824) | zero-pad to 5 digits | correct identifiers |

In [ ]:
df = raw.copy()
df.columns = df.columns.str.strip()

before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows")

for col in ["Order Date", "Ship Date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

for col in ["Sales", "Profit", "Discount", "Quantity"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.dropna(subset=["Order Date", "Sales", "Profit"])
print(f"Dropped {before - len(df)} rows missing Order Date / Sales / Profit")

df["Discount"] = df["Discount"].fillna(0)
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())

cat_cols = df.select_dtypes(include="object").columns
df[cat_cols] = df[cat_cols].apply(lambda s: s.str.strip()).fillna("Unknown")
if "Postal Code" in df:
    pc = pd.to_numeric(df["Postal Code"], errors="coerce")
    df["Postal Code"] = pc.apply(lambda v: f"{int(v):05d}" if pd.notna(v) else "Unknown")

# Feature engineering
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month Name"] = df["Order Date"].dt.strftime("%b")
df["Year-Month"] = df["Order Date"].dt.to_period("M")
df["Profit Margin"] = np.where(df["Sales"] != 0, df["Profit"] / df["Sales"], np.nan)
df["Ship Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Discount Band"] = pd.cut(df["Discount"], bins=[-0.001, 0, 0.1, 0.2, 0.3, 0.5, 1.0],
                             labels=["0%", "0-10%", "10-20%", "20-30%", "30-50%", "50%+"])

print("Clean shape:", df.shape, "| remaining nulls:", int(df.isna().sum().sum()))
df[["Order Date", "Year-Month", "Sales", "Profit", "Discount", "Discount Band"]].head()

## 4. Customer purchase volume

In [ ]:
cust = (df.groupby(["Customer ID", "Customer Name"])
          .agg(Orders=("Order ID", "nunique"), Sales=("Sales", "sum"), Profit=("Profit", "sum"))
          .reset_index())

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(cust["Orders"], bins=range(1, int(cust["Orders"].max()) + 2), ax=ax[0], color="#4C72B0")
ax[0].set(title="Distribution of orders per customer", xlabel="Number of orders", ylabel="Customers")

top10 = cust.nlargest(10, "Sales").sort_values("Sales")
ax[1].barh(top10["Customer Name"], top10["Sales"], color="#55A868")
ax[1].set(title="Top 10 customers by sales", xlabel="Sales ($)")
plt.tight_layout(); plt.show()

print(f"Customers: {len(cust):,} | avg orders/customer: {cust.Orders.mean():.1f} | "
      f"repeat customers (2+ orders): {(cust.Orders >= 2).mean():.1%}")

## 5. Monthly seasonality & sales velocity

In [ ]:
monthly = df.groupby("Year-Month")[["Sales", "Profit"]].sum()
monthly.index = monthly.index.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly.index, monthly["Sales"], marker="o", label="Sales", color="#4C72B0")
ax.plot(monthly.index, monthly["Profit"], marker="o", label="Profit", color="#C44E52")
ax.plot(monthly.index, monthly["Sales"].rolling(3).mean(), ls="--", color="gray", label="Sales 3-mo avg")
ax.set(title="Monthly sales velocity", ylabel="USD"); ax.legend()
plt.show()

order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
heat = df.pivot_table(index="Year", columns="Month Name", values="Sales", aggfunc="sum")[[m for m in order if m in df["Month Name"].unique()]]
plt.figure(figsize=(13, 3.5))
sns.heatmap(heat, annot=True, fmt=",.0f", cmap="YlGnBu", cbar_kws={"label": "Sales ($)"})
plt.title("Sales by year and month (seasonality heatmap)"); plt.show()

yearly = df.groupby("Year")[["Sales", "Profit"]].sum()
yearly["Sales YoY %"] = yearly["Sales"].pct_change() * 100
yearly

## 6. Category, Region and Segment performance

In [ ]:
def agg(by):
    g = df.groupby(by).agg(Sales=("Sales","sum"), Profit=("Profit","sum"),
                           Orders=("Order ID","nunique"), Avg_Discount=("Discount","mean"))
    g["Margin %"] = g["Profit"] / g["Sales"] * 100
    return g.sort_values("Sales", ascending=False)

for by in ["Category", "Region", "Segment"]:
    print(f"--- {by} ---"); display(agg(by))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, by in zip(axes, ["Category", "Region", "Segment"]):
    g = agg(by)[["Sales", "Profit"]]
    g.plot(kind="bar", ax=ax, color=["#4C72B0", "#55A868"], rot=0)
    ax.set_title(f"Sales & Profit by {by}"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

pivot = df.pivot_table(index="Category", columns="Region", values="Profit", aggfunc="sum")
plt.figure(figsize=(8, 3.5))
sns.heatmap(pivot, annot=True, fmt=",.0f", cmap="RdYlGn", center=0)
plt.title("Profit by Category x Region"); plt.show()

sub = agg(["Category", "Sub-Category"]).reset_index().sort_values("Profit")
plt.figure(figsize=(10, 7))
sns.barplot(data=sub, y="Sub-Category", x="Profit", hue="Category", dodge=False)
plt.axvline(0, color="black", lw=.8); plt.title("Profit by Sub-Category"); plt.show()

## 7. Discount sensitivity

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(data=df, x="Discount", y="Profit", hue="Category", alpha=.45, ax=ax[0])
ax[0].axhline(0, color="black", lw=.8); ax[0].set_title("Discount vs Profit (each dot = order line)")

band = df.groupby("Discount Band", observed=True).agg(Profit=("Profit","sum"), Sales=("Sales","sum"), Lines=("Profit","size"))
band["Margin %"] = band["Profit"] / band["Sales"] * 100
sns.barplot(x=band.index.astype(str), y=band["Profit"], ax=ax[1],
            palette=["#55A868" if v >= 0 else "#C44E52" for v in band["Profit"]])
ax[1].axhline(0, color="black", lw=.8); ax[1].set(title="Total profit by discount band", xlabel="Discount band")
plt.tight_layout(); plt.show()

print(f"Correlation (Discount vs Profit): {df['Discount'].corr(df['Profit']):.3f}")
print(f"Correlation (Discount vs Profit Margin): {df['Discount'].corr(df['Profit Margin']):.3f}")
band

## 8. Executive takeaways
The cell below writes each takeaway **from the actual computed numbers**, so the figures always match the dataset you ran.

In [ ]:
T = []
money = lambda x: f"-${abs(x):,.0f}" if x < 0 else f"${x:,.0f}"
tot_s, tot_p = df.Sales.sum(), df.Profit.sum()

# 1. Category revenue vs profitability
c = agg("Category")
top_c, low_m = c.index[0], c["Margin %"].idxmin()
best_m = c["Margin %"].idxmax()
T.append(f"**1. Category: revenue is not the same as profit.** {top_c} leads revenue ({money(c.loc[top_c,'Sales'])}, "
         f"{c.loc[top_c,'Sales']/tot_s:.0%} of total), but {best_m} has the best margin ({c.loc[best_m,'Margin %']:.1f}%) "
         f"while {low_m} has the weakest ({c.loc[low_m,'Margin %']:.1f}%). "
         f"*Action:* protect {best_m} pricing; review cost/discount structure in {low_m}.")

# 2. Seasonality
m = df.groupby("Month")["Sales"].sum()
top3 = m.nlargest(3).index.tolist()
names = {i: n for i, n in enumerate(order, 1)}
q4 = m.reindex([10, 11, 12]).sum() / m.sum()
T.append(f"**2. Sales are seasonal.** Sales peak in {', '.join(names[i] for i in top3)} (together {m[top3].sum()/m.sum():.0%} of sales); "
         f"Q4 alone delivers {q4:.0%}. The weakest month is {names[m.idxmin()]}. "
         f"*Action:* build inventory and staffing ahead of Q4; use promotions to lift the slow months.")

# 3. Growth
y = df.groupby("Year")["Sales"].sum()
if len(y) > 1:
    cagr = (y.iloc[-1] / y.iloc[0]) ** (1 / (len(y) - 1)) - 1
    T.append(f"**3. Growth trend.** Sales moved from {money(y.iloc[0])} in {y.index[0]} to {money(y.iloc[-1])} in {y.index[-1]} "
             f"(~{cagr:.0%} compound annual growth); the best year was {y.idxmax()}. "
             f"*Action:* check that profit grew with sales (see the yearly table in section 5).")

# 4. Discount sensitivity
hi, lo = df[df.Discount > 0.2], df[df.Discount <= 0.2]
verb = "destroy" if hi.Profit.sum() < 0 else "dilute"
T.append(f"**4. Discounts above 20% {verb} profit.** Lines with discount > 20% are {len(hi)/len(df):.0%} of all lines but total "
         f"{money(hi.Profit.sum())} profit, versus {money(lo.Profit.sum())} for the rest "
         f"(correlation of discount with profit = {df.Discount.corr(df.Profit):.2f}). "
         f"*Action:* cap discounts at ~20% unless approved.")

# 5. Region / state
r = agg("Region"); worst_r, best_r = r["Margin %"].idxmin(), r["Margin %"].idxmax()
st = df.groupby("State")["Profit"].sum().sort_values()
loss_states = st[st < 0]
T.append(f"**5. Regional gaps.** {best_r} has the best margin ({r.loc[best_r,'Margin %']:.1f}%) and {worst_r} the worst "
         f"({r.loc[worst_r,'Margin %']:.1f}%). "
         + (f"{len(loss_states)} states lose money overall, the worst being {', '.join(loss_states.index[:3])}. " if len(loss_states) else "No state is loss-making overall. ")
         + f"*Action:* investigate discounting and product mix in those states.")

# 6. Loss-making sub-categories
sc = df.groupby("Sub-Category")["Profit"].sum().sort_values()
neg = sc[sc < 0]
if len(neg):
    T.append(f"**6. Loss-making sub-categories.** {', '.join(neg.index)} lose money in total ({money(neg.sum())}); "
             f"the largest profit contributor is {sc.index[-1]} ({money(sc.iloc[-1])}). "
             f"*Action:* reprice or cut discounts on the loss-makers.")
else:
    T.append(f"**6. Sub-category profit.** All sub-categories are profitable; the top earner is {sc.index[-1]} ({money(sc.iloc[-1])}).")

# 7. Customers & segments
s = agg("Segment"); topn = max(1, int(len(cust) * .1))
share = cust.nlargest(topn, "Sales").Sales.sum() / cust.Sales.sum()
T.append(f"**7. Customers & segments.** {s.index[0]} is the largest segment ({s.iloc[0]['Sales']/tot_s:.0%} of sales). "
         f"The top 10% of customers generate {share:.0%} of revenue, and {(cust.Orders>=2).mean():.0%} of customers are repeat buyers "
         f"(avg {cust.Orders.mean():.1f} orders). *Action:* loyalty/retention offers for top accounts.")

display(Markdown(f"### Overall: {money(tot_s)} sales | {money(tot_p)} profit | {tot_p/tot_s:.1%} margin | {df['Order ID'].nunique():,} orders\n\n" + "\n\n".join(T)))

## Conclusion & recommendations
1. **Fix discount policy first:** deep discounts are the single biggest profit leak.
2. **Plan for seasonality:** concentrate stock, marketing and staffing on the Q4 peak.
3. **Mix matters more than volume:** shift emphasis toward high-margin categories/sub-categories.
4. **Target weak regions/states** with pricing and shipping reviews.
5. **Retain top customers**, since a small share of customers drives most revenue.

*Limitations:* historical data only; profit is shown at order-line level, so costs outside the dataset (marketing, overhead) are not included.